# 05 — Training, validation, and standalone CLI

> **Status:** empty implementation skeleton.

- **Mapped issue:** [#12](https://github.com/majorgilles/transformer-2017-reproduction/issues/12)
- **Depends on:** `04_objective_optimizer_batching.ipynb` / issue #11.


In [ ]:
#| default_exp training


## Goal

Deliver notebook-independent fixture training and validation.


## One training pass

In [ ]:
from collections.abc import Sequence

import torch

from transformer_2017_reproduction.model import Transformer
from transformer_2017_reproduction.optimization import label_smoothed_loss

In [ ]:
#| export
def train(
    model: Transformer,
    batches: Sequence[tuple[torch.Tensor, torch.Tensor]],
    optimizer: torch.optim.Optimizer,
) -> float:
    if len(batches) == 0:
        raise ValueError("Cannot compute loss on empty sequence of batches.")

    total_loss = 0.0
    model.train()
    for source_token_ids, target_token_ids in batches:
        optimizer.zero_grad()

        decoder_input_ids = target_token_ids[:, :-1]
        logits = model(
            source_token_ids=source_token_ids,
            target_token_ids=decoder_input_ids,
        )

        next_token_labels = target_token_ids[:, 1:]
        loss = label_smoothed_loss(
            logits=logits,
            targets=next_token_labels,
            pad_token_id=model.pad_token_id,
        )
        loss.backward()

        optimizer.step()
        total_loss += loss.item()

    return total_loss / len(batches)

## Required deliverables

- Typed trainer state/metrics
- Train/validate CLI
- Validation ranking
- Deterministic learning curves


## Planned implementation sections

1. Paper and contract references
2. Typed implementation
3. Focused tests
4. Deterministic visible result
5. Exported API and artifact identities


## Explicitly deferred

Resume format, WMT pipeline, canonical run, and final test.


## HITL checkpoint

Approve CLI behavior, learning curves, validation boundaries, and config.
